## Section 0 - Configuration

Marker characters used throughout this tokenizer. We use rare Unicode characters rather than common symbols like `_` to avoid colliding with characters that may legitimately appear in real text or code (for example, `_` is a normal character in Python variable names like `word_freqs`).

`END_OF_WORD` follows the same convention used by SentencePiece, a real-world tokenizer. `NO_SPACE` uses Unicode's WORD JOINER character, designed for exactly the "no space here" purpose we need when splitting punctuation away from words without losing the original spacing on decode.

Set (`NO_SPACE_DEBUG_MODE` = False, `END_OF_WORD_DEBUG_MODE` = False) to use SentencePiece convention and Unicode's WORD JOINER character.

Set (`NO_SPACE_DEBUG_MODE` = True, `END_OF_WORD_DEBUG_MODE` = True) to use human-visible substitute characters, for easier inspection of printed output during development.

For readability of the example code output and results documentation, the code was run using (`NO_SPACE_DEBUG_MODE` = True, `END_OF_WORD_DEBUG_MODE` = False). Each flag is independent. END_OF_WORD_DEBUG_MODE rarely needs to be True, since ▁ already renders clearly. NO_SPACE_DEBUG_MODE is more often useful as True, since the real WORD JOINER character prints as an unreadable escape code rather than rendering invisibly.

In [1]:
# Set to True to use a human-visible substitute character instead of the
# real, invisible Unicode WORD JOINER - useful when printing/inspecting
# output, since \u2060 displays as an unreadable escape code rather than
# rendering invisibly the way it does in actual text.
NO_SPACE_DEBUG_MODE = True

# END_OF_WORD (▁) already renders clearly as a real character, so there's
# normally no need to substitute it - kept as a separate toggle in case
# that ever changes (e.g. a font/environment where it doesn't render).
END_OF_WORD_DEBUG_MODE = False

if END_OF_WORD_DEBUG_MODE:
    END_OF_WORD = '\u00a7'   # '§' - human-visible substitute for debugging
else:
    END_OF_WORD = '\u2581'   # '▁' - same marker convention used by SentencePiece

if NO_SPACE_DEBUG_MODE:
    NO_SPACE = '\u00a4'      # '¤' - human-visible substitute; \u2060 prints
                              # as an escape code rather than rendering
                              # invisibly, making merge lists hard to scan
else:
    NO_SPACE = '\u2060'      # Unicode WORD JOINER - the real, invisible
                              # marker, appropriate for actual encode/decode
                              # use outside of human-readable inspection

import string

## Section 1 - Training Functions

These functions handle learning the BPE vocabulary from a piece of text: 
- spliting punctuations off from text
- counting word frequencies
- representing words as symbols
- counting pair frequencies
- iteratively merging the most common pairs

In [2]:
def split_punctuation(text):
    """
    Split a single word into two pieces at the first punctuation
    character found, if any. Returns (need_split, left_split, right_split).
    """
    need_split = False
    left_split = ""
    right_split = ""

    if len(text) > 1:
    
        if text[0] == NO_SPACE:
            # NO_SPACE is the first character
            first_punct_idx = next((i for i, char in enumerate(text) if char in string.punctuation), None)
            if first_punct_idx == None:
                # No punctuation found, no need to do anything, returns need_split = False
                pass
            else:
                if first_punct_idx == 1:
                    if len(text) > 2:
                        #starts with a NO_SPACE followed by punctuation, split occurs after punctuation
                        left_split = text[:2]
                        right_split = NO_SPACE + text[2:]
                        need_split = True
                    else:
                        #NO_SPACE followed by punctuation, no need to split
                        pass
                else:
                    #punctuation found elsewhere, split occurs before punctuation
                    left_split = text[:first_punct_idx]
                    right_split = NO_SPACE + text[first_punct_idx:]
                    need_split = True  
    
        else:
            # NO_SPACE is not the first character
            first_punct_idx = next((i for i, char in enumerate(text) if char in string.punctuation), None)
            if first_punct_idx == None:
                # No punctuation found, no need to do anything, returns need_split = False
                pass
            else:
                if first_punct_idx == 0:
                    #starts with a punctuation, split occurs after punctuation
                    left_split = text[0]
                    right_split = NO_SPACE + text[1:]
                    need_split = True
                else:
                    #punctuation found elsewhere, split occurs before punctuation
                    left_split = text[:first_punct_idx]
                    right_split = NO_SPACE + text[first_punct_idx:]
                    need_split = True
    else:
        #length is not greater than 1, no need to split
        pass




    return need_split, left_split, right_split

In [3]:
def split_all_punctuation(word):
    """
    Repeatedly apply split_punctuation to a word until no further
    splitting is needed, collecting all resulting pieces in order.
    """
    pieces = []
    remaining = word
    
    need_split, left_split, right_split = split_punctuation(remaining)
    
    while need_split:
        pieces.append(left_split)
        remaining = right_split
        need_split, left_split, right_split = split_punctuation(remaining)
    
    pieces.append(remaining)  # whatever's left when no more splitting is needed
    
    return pieces

In [4]:
def get_word_freqs(text):
    """
    Split text into words and count how often each word appears.
    Returns a dict like {'low': 3, 'lower': 1, 'lowest': 1}
    """
    if END_OF_WORD in text or NO_SPACE in text:
        print("WARNING: input text already contains a reserved marker character "
              "(END_OF_WORD or NO_SPACE). This tokenizer assumes these characters "
              "never appear in raw input - results may be incorrect.")

    raw_words = text.split()
    processed_words = []
    
    for word in raw_words:
        processed_words.extend(split_all_punctuation(word))
    
    word_freqs = {}
    for word in processed_words:
        word_freqs[word] = word_freqs.get(word, 0) + 1
    return word_freqs

In [5]:
def merge_freq_dicts(master, new, stats=None):
    """
    Merge a new word_freqs dictionary into an existing master dictionary,
    in place. Modifies master directly; does not return a new dictionary.
    
    Raises TypeError if either dictionary doesn't have the expected
    shape (string keys, integer values) - this is a user-facing function,
    so inputs aren't assumed to be well-formed.
    """
    if not all(type(k) == str and type(v) == int for k, v in master.items()):
        raise TypeError("master dictionary must have string keys and integer values")
    if not all(type(k) == str and type(v) == int for k, v in new.items()):
        raise TypeError("new dictionary must have string keys and integer values")
    
    for key, count in new.items():
        if stats is not None:
            if master.get(key, 0) == 0:
                stats['new'] = stats.get('new', 0) + 1
            else:
                stats['old'] = stats.get('old', 0) + 1
        master[key] = master.get(key, 0) + count

In [6]:
def words_to_symbols(word_freqs):
    """
    Convert each word into a list of characters + end-of-word marker.
    Returns a dict like {('l','o','w', END_OF_WORD): 3, ('l','o','w','e','r', END_OF_WORD): 1, ...}
    We use a tuple of symbols as the key (tuples are hashable, lists aren't).
    """
    symbol_freqs = {}
    for word, freq in word_freqs.items():
        symbols = tuple(list(word) + [END_OF_WORD])
        symbol_freqs[symbols] = freq
    return symbol_freqs

In [7]:
def get_pair_freqs(symbol_freqs):
    """
    Count how often each adjacent pair of symbols appears,
    weighted by word frequency.
    Returns a dict like {('l','o'): 5, ('o','w'): 5, ...}
    """
    pair_freqs = {}
    for symbols, freq in symbol_freqs.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq
    return pair_freqs

In [8]:
def get_best_pair(pair_freqs):
    """
    Find the pair with the highest frequency.
    Returns None if there are no pairs left.
    """
    if not pair_freqs:
        return None
    best_pair = max(pair_freqs, key=pair_freqs.get)
    return best_pair

In [9]:
def merge_pair(symbol_freqs, pair_to_merge):
    """
    Replace every occurrence of pair_to_merge with a single merged symbol,
    across all words in symbol_freqs.
    """
    new_symbol_freqs = {}
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]  # e.g. 'l' + 'o' -> 'lo'

    for symbols, freq in symbol_freqs.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            # Check if the pair starting at position i matches what we're merging
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
                new_symbols.append(merged_symbol)
                i += 2   # skip both symbols we just merged
            else:
                new_symbols.append(symbols[i])
                i += 1   # just move forward normally

        new_symbol_freqs[tuple(new_symbols)] = freq

    return new_symbol_freqs

In [10]:
def train_bpe(text, target_vocab_size):
    """
    Train a BPE tokenizer on the given text.
    Returns:
        merges: list of merge rules in the order they were learned
        vocab: set of all symbols in the final vocabulary
    """
    word_freqs = get_word_freqs(text)
    symbol_freqs = words_to_symbols(word_freqs)

    # Start vocab with every individual character we've seen, plus the end marker
    vocab = set()
    for symbols in symbol_freqs:
        vocab.update(symbols)

    merges = []  # we'll record merge rules in order here

    while len(vocab) < target_vocab_size:
        pair_freqs = get_pair_freqs(symbol_freqs)
        best_pair = get_best_pair(pair_freqs)

        if best_pair is None:
            print("No more pairs to merge — stopping early.")
            break

        symbol_freqs = merge_pair(symbol_freqs, best_pair)
        merges.append(best_pair)

        merged_symbol = best_pair[0] + best_pair[1]
        vocab.add(merged_symbol)

    return merges, vocab

In [11]:
def train_bpe_multi(file_list, target_vocab_size, stats_interval = None):
    """
    Train BPE across many source files without holding all raw text
    in memory simultaneously. Processes one file at a time into a
    word_freqs dictionary, merging into a running master as it goes.
    If one file fails, it's skipped (flagged immediately) and processing
    continues with the remaining files - completed work isn't lost.
    """
    interval_stats = []
    vocab = set()
    merges = []

    total_books = len(file_list)
    book_counter = 0
    current_stats = {'new': 0, 'old': 0}
    master_freqs = {}

    for filepath in file_list:
        try:
            book_counter += 1
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            file_word_freqs = get_word_freqs(text)
            
            if stats_interval is not None:
                # always collect stats when tracking is enabled
                merge_freq_dicts(master_freqs, file_word_freqs, current_stats)
                # check if we've hit a checkpoint
                if book_counter % stats_interval == 0 or book_counter == total_books:
                    if (current_stats['new'] + current_stats['old'] == 0):
                        new_percentage = -1  # sentinel: no books successfully processed in this interval
                    else:
                        new_percentage = current_stats['new'] / (current_stats['new'] + current_stats['old'])
                    interval_stats.append((book_counter, new_percentage))
                    current_stats['new'] = 0
                    current_stats['old'] = 0
            else:
                # tracking disabled, merge without stats
                merge_freq_dicts(master_freqs, file_word_freqs, None)
                
        except Exception as e:
            print(f"WARNING: skipping '{filepath}' due to error: {e}")
            continue
    
    symbol_freqs = words_to_symbols(master_freqs)

    for symbols in symbol_freqs:
        vocab.update(symbols)

    while len(vocab) < target_vocab_size:
        pair_freqs = get_pair_freqs(symbol_freqs)
        best_pair = get_best_pair(pair_freqs)
        if best_pair is None:
            print("No more pairs to merge — stopping early.")
            break
        symbol_freqs = merge_pair(symbol_freqs, best_pair)
        merges.append(best_pair)
        merged_symbol = best_pair[0] + best_pair[1]
        vocab.add(merged_symbol)
    
    return merges, vocab, interval_stats

## Section 2 - Encoding/decoding functions

These functions handle the application of the learned BPE vocabulary:
- applying a single merge rule
- encoding a single word based on all rules
- decoding the symbols back into a single word
- encoding an entire text
- decoding a series of encoded text

In [12]:
def apply_merge(symbols, pair_to_merge):
    """
    Apply a single merge rule to one word's tuple of symbols.
    Returns a new tuple with the pair merged wherever it appears.
    """
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]
    new_symbols = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
            new_symbols.append(merged_symbol)
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return tuple(new_symbols)

In [13]:
def encode_word(word, merges):
    """
    Encode a single word using the learned merge rules, in order.
    """
    symbols = tuple(list(word) + [END_OF_WORD])
    for pair in merges:
        symbols = apply_merge(symbols, pair)
    return symbols

In [14]:
def decode(symbols):
    """
    Convert a tuple of symbols back into a readable string.
    
    Includes a runtime check verifying that every NO_SPACE marker gets
    consumed during END_OF_WORD + NO_SPACE cleanup, as designed. If this
    ever fires, it means a NO_SPACE marker was created in a context the
    splitting logic didn't anticipate - worth investigating, not ignoring.
    
    In a production pipeline this check would route through a proper
    logging system rather than print(); a print statement is appropriate
    here given this is a notebook, not a deployed service.
    """
    text = "".join(symbols)
    text = text.replace(END_OF_WORD + NO_SPACE, "")
    
    leftover_no_space_count = text.count(NO_SPACE)
    if leftover_no_space_count > 0:
        print(f"WARNING: {leftover_no_space_count} unexpected leftover NO_SPACE marker(s) found in decode. This should not happen - investigate.")
    
    text = text.replace(END_OF_WORD, " ")
    return text.strip()

In [15]:
def encode_text(text, merges):
    """
    Encode a full piece of text (multiple words) using learned merge rules.
    Returns a list of symbol-tuples, one per word.
    """
    words = text.split()
    return [encode_word(word, merges) for word in words]

In [16]:
def decode_text(encoded_words):
    """
    Decode a list of symbol-tuples (one per word) back into a sentence.
    """
    words = [decode(symbols) for symbols in encoded_words]
    return " ".join(words)

## Section 3 - Demonstrating and testing

Before applying this tokenizer to real text, we verify correctness on a small, fully traceable example: a 5-word toy vocabulary ("low", "low", "low", "lower", "lowest"). Every merge step below was predicted by hand before running the code. The encode/decode tests include words that were not seen during training to confirm the tokenizer generalizes well when it encounters something unfamiliar.

Toy training data and full hand-traced walkthrough:

In [17]:
text = "low low low lower lowest"
word_freqs = get_word_freqs(text)
print(word_freqs)

symbol_freqs = words_to_symbols(word_freqs)
print(symbol_freqs)

{'low': 3, 'lower': 1, 'lowest': 1}
{('l', 'o', 'w', '▁'): 3, ('l', 'o', 'w', 'e', 'r', '▁'): 1, ('l', 'o', 'w', 'e', 's', 't', '▁'): 1}


In [18]:
pair_freqs = get_pair_freqs(symbol_freqs)
print(pair_freqs)

{('l', 'o'): 5, ('o', 'w'): 5, ('w', '▁'): 3, ('w', 'e'): 2, ('e', 'r'): 1, ('r', '▁'): 1, ('e', 's'): 1, ('s', 't'): 1, ('t', '▁'): 1}


In [19]:
best_pair = get_best_pair(pair_freqs)
print(best_pair)

('l', 'o')


In [20]:
new_symbol_freqs = merge_pair(symbol_freqs, best_pair)
print(new_symbol_freqs)

{('lo', 'w', '▁'): 3, ('lo', 'w', 'e', 'r', '▁'): 1, ('lo', 'w', 'e', 's', 't', '▁'): 1}


Full training run on the toy example:

In [21]:
text = "low low low lower lowest"
merges, vocab = train_bpe(text, target_vocab_size=20)

print("Merges learned, in order:")
for m in merges:
    print(m)

print("\nFinal vocab:")
print(vocab)

No more pairs to merge — stopping early.
Merges learned, in order:
('l', 'o')
('lo', 'w')
('low', '▁')
('low', 'e')
('lowe', 'r')
('lower', '▁')
('lowe', 's')
('lowes', 't')
('lowest', '▁')

Final vocab:
{'▁', 'e', 'low', 'low▁', 'l', 'r', 'lowest▁', 'w', 'lo', 'lowes', 'lowe', 'lower', 'lower▁', 'lowest', 't', 'o', 's'}


Testing encode on known and unseen words:

In [22]:
print(encode_word("slow", merges))
print(encode_word("loan", merges))
print(encode_word("lime", merges))
print(encode_word("slower", merges))

('s', 'low▁')
('lo', 'a', 'n', '▁')
('l', 'i', 'm', 'e', '▁')
('s', 'lower▁')


Testing decode (round trip verification):

In [23]:
print(decode(('s', 'low_')))
print(decode(('lo', 'a', 'n', '_')))
print(decode(('l', 'i', 'm', 'e', '_')))
print(decode(('s', 'lower_')))

slow_
loan_
lime_
slower_


Testing full sentence encode/decode:

In [24]:
test_text = "low lower lime"
encoded = encode_text(test_text, merges)
print(encoded)

decoded = decode_text(encoded)
print(decoded)

[('low▁',), ('lower▁',), ('l', 'i', 'm', 'e', '▁')]
low lower lime


Testing `split_all_punctuation` and `get_word_freqs` on edge cases: consecutive punctuation, multiple punctuation marks within one word, and a code fragment

In [25]:
get_word_freqs("apple app'le ap%pl^e wait... really?! wait... really?! symbols = tuple(list(word) + ['_'])")

{'apple': 1,
 'app': 1,
 "¤'": 3,
 '¤le': 1,
 'ap': 1,
 '¤%': 1,
 '¤pl': 1,
 '¤^': 1,
 '¤e': 1,
 'wait': 2,
 '¤.': 6,
 'really': 2,
 '¤?': 2,
 '¤!': 2,
 'symbols': 1,
 '=': 1,
 'tuple': 1,
 '¤(': 2,
 '¤list': 1,
 '¤word': 1,
 '¤)': 2,
 '+': 1,
 '[': 1,
 '¤_': 1,
 '¤]': 1}

### What this confirmed

The hand-traced predictions matched the code's output exactly, including an instructive edge case: because this toy vocabulary only contains 3 unique words, the algorithm eventually runs out of meaningful frequency signal and starts merging pairs that only occur once. "lower" and "lowest" end up fully merged into single whole-word tokens - not because they are common English words, but because by that point only tied pairs at frequency = 1 remained, and the algorithm had nothing left to meaningfully compare them against. This is a useful early warning sign for what we would expect to see more clearly at larger scale: a tokenizer's vocabulary is entirely a reflection of its training data's statistics, not of any linguistic notion of what "deserves" to be a token.

The encode test on "slower" (a word combining a known prefix with an unseen full word) and "lime" (sharing no merge patterns with training data at all) confirms the tokenizer generalizes well, falling back to individual characters rather than failing when it meets something outside its training vocabulary.

## Section 4 - Real-world Application - English Prose

Training on a 5-word toy example confirms the algorithm works correctly, but real text reveals patterns - and limitations - that don't appear at small scale. This section trains the tokenizer on the full text of *Alice's Adventures in Wonderland*, a public domain book from Project Gutenberg, to see what a tokenizer trained on real English prose actually learns.

Opening and storing the text:

In [26]:
with open("alice.txt", "r", encoding="utf-8") as f:
    alice_text = f.read()

print(len(alice_text))       # total characters
print(alice_text[:300])      # peek at the beginning

144696
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    T


Using start and end markers to extract the clean text content in between: 

The actual header is "*** START OF THE PROJECT GUTENBERG EBOOK 11 ***". 

Due to other Project Gutenberg Ebooks having different identification numbers we search for the trailing "***" to determine where to start the slice.

In [27]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
marker_search_start = alice_text.find(start_marker)
header_end = alice_text.find("***", marker_search_start + len(start_marker))
content_start = header_end + 3
end_index = alice_text.find(end_marker)

clean_text = alice_text[content_start : end_index]
print(len(clean_text))
print(clean_text[:200])

144603


[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER I


Training the BPE with the extracted English prose text:

In [28]:
merges, vocab = train_bpe(clean_text, target_vocab_size=500)
print(f"Number of merges learned: {len(merges)}")
print(f"Final vocab size: {len(vocab)}")

Number of merges learned: 425
Final vocab size: 500


In [29]:
print("First 20 merges learned:")
for m in merges[:20]:
    print(m)

print("\nLast 20 merges learned:")
for m in merges[-20:]:
    print(m)

First 20 merges learned:
('e', '▁')
('t', '▁')
('t', 'h')
('d', '▁')
('¤', ',')
('¤,', '▁')
('s', '▁')
('i', 'n')
('e', 'r')
('o', 'u')
('th', 'e▁')
('a', 'n')
('y', '▁')
('o', '▁')
('”', '▁')
('n', '▁')
('¤', '”▁')
('g', '▁')
('¤', '.')
('¤.', '▁')

Last 20 merges learned:
('be', 'en▁')
('ba', 'ck▁')
('h', 'ear')
('“', 'S')
('M', 'ar')
('be', 'fore▁')
('af', 'ter▁')
('s', 'ti')
('B', 'ut▁')
('fro', 'm▁')
('p', 'e')
('e', 'v')
('¤', '—')
('fu', 'l▁')
('en', 'ce▁')
('tt', 'ing▁')
('c', 'ur')
('ou', 'ld')
('t', 'u')
('da', 'y▁')


### What this revealed
### Version 1:

The early merges are dominated by extremely common English patterns, including not only letter pairs like `('t','h')`, but also word-boundary patterns like `('e','_')`, reflecting how many English words end in "e". Notably, "the" (the most common word in English) assembles itself piece by piece purely from frequency: `('t','h')` merges early, `('e','_')` merges separately, and eventually `('th','e_')` combines them into a single token representing the whole word "the ".

This also surfaces a real limitation worth naming directly: because word splitting here is based only on whitespace, punctuation stays attached to words. Merges like `('i','s,_')` treat "is," as an entirely different token from "is", despite being grammatically the same word. This is a deliberate finding, not an oversight. It is documented as a known limitation and addressed as a "Version 2" improvement (see version2_goals.md), which separates punctuation from words before BPE training begins.

### Version 2: results with punctuation pre-tokenization

Re-running training on the same Alice in Wonderland text, now with punctuation split into its own standalone tokens before BPE begins, directly resolves the limitation identified in Version 1. The 5th and 6th top merges are `('¤', ',')` and `('¤,', '▁')` showing the comma merging as its own free-standing token. Compare this to version 1 where the chain of merges ending in`('i', 's,_')` shows 's' and ',' fused together at an earlier step, leading to "is," and "is" being treated as entirely unrelated tokens.

The same core English patterns still emerge as expected: `('t','h')`, `('e', '▁')`, and "the" still assembling piece by piece, with `('th', 'e▁')` combining both pieces into a single token. This confirms the fix didn't disrupt the genuine, meaningful merges - it only removed the punctuation artifacts sitting alongside them. The last 20 merges now show clean, recognizable whole words ("been", "back", "before", "after", "from", "day") independent from any potential adjacent punctuation. This provides a cleaner and more meaningful picture when compared to the Version 1 results.

## Section 5 - Comparison Case - Python Code

Code and natural language prose have different statistical structure: code has more unique, project-specific identifiers (variable names chosen by an author) and a narrower set of highly repeated keywords and syntax characters. This section tests that hypothesis directly by training the same tokenizer on a small Python code sample - three of the functions defined in Section 1 - using a smaller target vocabulary size, since the sample itself is much shorter than a full novel.

Example text taken from this project's Python code:

In [30]:
python_code = '''
def get_word_freqs(text):
    words = text.split()
    word_freqs = {}
    for word in words:
        word_freqs[word] = word_freqs.get(word, 0) + 1
    return word_freqs


def words_to_symbols(word_freqs):
    symbol_freqs = {}
    for word, freq in word_freqs.items():
        symbols = tuple(list(word) + ['_'])
        symbol_freqs[symbols] = freq
    return symbol_freqs


def get_pair_freqs(symbol_freqs):
    pair_freqs = {}
    for symbols, freq in symbol_freqs.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq
    return pair_freqs
'''

Observing the first 20 split words for examples and identifying patterns:

In [31]:
words = python_code.split()
print(words[:20])

['def', 'get_word_freqs(text):', 'words', '=', 'text.split()', 'word_freqs', '=', '{}', 'for', 'word', 'in', 'words:', 'word_freqs[word]', '=', 'word_freqs.get(word,', '0)', '+', '1', 'return', 'word_freqs']


Training the BPE with the Python code example:

In [32]:
code_merges, code_vocab = train_bpe(python_code, target_vocab_size=150)
print(f"Number of merges learned: {len(code_merges)}")
print(f"Final vocab size: {len(code_vocab)}")

print("\nFirst 15 merges:")
for m in code_merges[:15]:
    print(m)

print("\nLast 15 merges:")
for m in code_merges[-15:]:
    print(m)

No more pairs to merge — stopping early.
Number of merges learned: 99
Final vocab size: 137

First 15 merges:
('s', '▁')
('r', 'e')
('¤', '_')
('¤_', '▁')
('f', 're')
('fre', 'q')
('o', 'r')
('¤', 'freq')
('¤freq', 's▁')
('w', 'or')
('wor', 'd')
('(', '▁')
('¤', ')')
('¤)', '▁')
('word', '▁')

Last 15 merges:
('tu', 'pl')
('tupl', 'e')
('tuple', '▁')
('¤l', 'i')
('¤li', 's')
('¤lis', 't▁')
('¤', 'symbol▁')
('i', '▁')
('r', 'a')
('ra', 'n')
('ran', 'ge')
('range', '▁')
('¤l', 'e')
('¤le', 'n▁')
('-', '▁')


### What this revealed
### Version 1:

The comparison confirms the hypothesis. The earliest merges quickly assemble recognizable fragments (`freq`, `word`, and `symbol`), reflecting how often `symbol_freqs`, `word_freqs`, and `pair_freqs` repeat throughout this short sample. If we had used a wide variety of Python code, by many different users, on many different applications, perhaps more common Python commands and syntax would have a higher frequency. 

The algorithm runs out of genuine pattern diversity well before reaching the target vocabulary size. By the final few merges, the tokenizer is reconstructing entire fragments of source code character-by-character: the literal sequence `['_'])` (drawn directly from this project's own `words_to_symbols` function) is rebuilt across four consecutive merges near the end of training. 

This is the same "running out of meaningful frequency signal" behavior observed in Section 3 with "lower" and "lowest". It illustrates why training data size and diversity matter relative to target vocabulary size: a tokenizer trained on a handful of similar files would overfit to that exact code, producing a vocabulary unlikely to generalize well to other Python code with different naming conventions and structure.

The same "Version 2" improvement noted in Section 4 of separating punctuation and special characters before BPE training may matter even more for code than for prose. Some coders may mitigate this with white space for readability, but not all. Code contains many more structurally meaningful symbols (`(`, `)`, `[`, `]`, `:`, `,`, `=`) than English text does, and the runaway merging seen here suggests that without separating them out first, a code-trained tokenizer risks memorizing exact syntax patterns rather than learning generalizable subword units.

### Version 2: results with punctuation pre-tokenization

Re-running training on the same Python code sample, with punctuation pre-tokenization in place, produces a clear, measurable improvement. The runaway merging seen in Version 1 — where entire function signatures collapsed into single tokens like ('get_pair_freqs(symbol_freqs', '):_') — does not happen here. The last 15 merges instead show complete, recognizable identifiers including `tuple`, `list`, `range`, `len`, and `symbol` - each capped with a clean end-of-word marker, not fused to surrounding syntax.

Training also stopped early this time ("No more pairs to merge") at 137 tokens, short of the 150 target, rather than continuing to artificially manufacture larger merges by gluing punctuation into the surrounding code the way Version 1 did. Running out of genuine pattern diversity sooner, and simply stopping, is healthier behavior than v1's alternative of continuing to "find" merges by absorbing structurally meaningless adjacency between code and punctuation.

One specific finding worth noting: the very first merge learned was `('s', '▁')` meaning a word ending in "s" was the single most frequent pattern in this sample, ahead of any letter-pair merge. This likely reflects how often `word_freqs`, `symbol_freqs`, and `pair_freqs` (all ending in "s") repeat throughout this small file. This is a reminder that even with punctuation properly separated, a small and narrow sample still produces a vocabulary shaped heavily by this specific file's naming conventions, not by Python in general.

## Section 6 - Multi-Source Training Experiment

### Motivation

A tokenizer trained on a single source overfits to that source's vocabulary and patterns, as demonstrated in Section 5. Real tokenizers are trained on large, diverse corpora (many different texts, authors, and styles). This section tests the multi-source training capability built in Version 2 (`train_bpe_multi`) and investigates a specific hypothesis: **does new-word discovery taper off as more sources are added?**

### Corpus

Ten public domain texts from Project Gutenberg were selected, spanning over 1,500 years (from 5th century to 1908) of written English. The corpus intentionally includes known overlaps: Romeo and Juliet appears within Shakespeare's complete works as well as in two separate standalone files - one modern-spelling edition, and one First Folio transcription preserving 17th-century printed spelling conventions.

| File | Work | Author | Original Date |
|---|---|---|---|
| shakespeare_clean.txt | Complete Works of William Shakespeare | William Shakespeare | 1590s–1613 |
| middlemarch_clean.txt | Middlemarch | George Eliot | 1871 |
| city_clean.txt | The City of God | Augustine of Hippo | 413–426 AD (trans. 1871) |
| moby_clean.txt | Moby-Dick | Herman Melville | 1851 |
| pride_clean.txt | Pride and Prejudice | Jane Austen | 1813 |
| frankenstein_clean.txt | Frankenstein | Mary Shelley | 1818 |
| room_clean.txt | A Room with a View | E.M. Forster | 1908 |
| alice_clean.txt | Alice's Adventures in Wonderland | Lewis Carroll | 1865 |
| romeo_clean.txt | The Tragedy of Romeo and Juliet | William Shakespeare | 1597 |
| romeo2_clean.txt | The Tragedie of Romeo and Juliet (First Folio) | William Shakespeare | 1623 |

The overlaps were kept deliberately to observe how the tokenizer handles real-world corpus characteristics including duplicate content, multiple editions of the same work, and content from dramatically different domains and eras.

### Setup

Two runs were performed: one in descending file size order, one in a randomized order. Both used `stats_interval=1` to track new-word discovery rate per source (the fraction of each source's words that were genuinely new to the accumulated vocabulary at the time it was processed).


In [33]:
merges, vocab, interval_stats = train_bpe_multi([
    "shakespeare_clean.txt", 
    "middlemarch_clean.txt", 
    "city_clean.txt", 
    "moby_clean.txt", 
    "pride_clean.txt", 
    "frankenstein_clean.txt", 
    "room_clean.txt", 
    "alice_clean.txt", 
    "romeo_clean.txt", 
    "romeo2_clean.txt"
], 500, stats_interval=1)

print(f"Number of merges learned: {len(merges)}")
print(f"Final vocab size: {len(vocab)}")

print("First 20 merges learned:")
for m in merges[:20]:
    print(m)

print("\nLast 20 merges learned:")
for m in merges[-20:]:
    print(m)

print(interval_stats)

Number of merges learned: 332
Final vocab size: 500
First 20 merges learned:
('e', '▁')
('t', 'h')
('s', '▁')
('t', '▁')
('d', '▁')
(',', '▁')
('¤', ',▁')
('n', '▁')
('.', '▁')
('¤', '.▁')
('e', 'r')
('y', '▁')
('o', 'u')
('i', 'n')
('a', 'n')
('o', '▁')
('th', 'e▁')
('o', 'r')
('f', '▁')
('a', 'r')

Last 20 merges learned:
('m', 'b')
('r', 'y▁')
('g', 're')
('u', 'p')
('so', 'me▁')
('on', 'g▁')
('m', 'en▁')
('w', 'ay▁')
('’', '▁')
('the', 'n▁')
('k', 'ing▁')
('co', 'me▁')
('n', 'i')
('o', 'ther▁')
('s', 'c')
('sh', 'ould▁')
('en', 'd▁')
('d', 'ed▁')
('l', 'or')
('T', '▁')
[(1, 1.0), (2, 0.5116789778398882), (3, 0.3458348103509394), (4, 0.3794257291431155), (5, 0.1317685047193244), (6, 0.08793324775353016), (7, 0.21521942110177406), (8, 0.1497987349051179), (9, 0.0), (10, 0.40223704022370405)]


In [34]:
merges, vocab, interval_stats = train_bpe_multi([
    'shakespeare_clean.txt',
    'frankenstein_clean.txt',
    'moby_clean.txt',
    'romeo2_clean.txt',
    'middlemarch_clean.txt',
    'city_clean.txt',
    'romeo_clean.txt',
    'pride_clean.txt',
    'room_clean.txt',
    'alice_clean.txt'
], 500, stats_interval=1)

print(f"Number of merges learned: {len(merges)}")
print(f"Final vocab size: {len(vocab)}")

print("First 20 merges learned:")
for m in merges[:20]:
    print(m)

print("\nLast 20 merges learned:")
for m in merges[-20:]:
    print(m)

print(interval_stats)

Number of merges learned: 332
Final vocab size: 500
First 20 merges learned:
('e', '▁')
('t', 'h')
('s', '▁')
('t', '▁')
('d', '▁')
(',', '▁')
('¤', ',▁')
('n', '▁')
('.', '▁')
('¤', '.▁')
('e', 'r')
('y', '▁')
('o', 'u')
('i', 'n')
('a', 'n')
('o', '▁')
('th', 'e▁')
('o', 'r')
('f', '▁')
('a', 'r')

Last 20 merges learned:
('m', 'b')
('r', 'y▁')
('g', 're')
('u', 'p')
('so', 'me▁')
('on', 'g▁')
('m', 'en▁')
('w', 'ay▁')
('’', '▁')
('the', 'n▁')
('k', 'ing▁')
('co', 'me▁')
('n', 'i')
('o', 'ther▁')
('s', 'c')
('sh', 'ould▁')
('en', 'd▁')
('d', 'ed▁')
('l', 'or')
('T', '▁')
[(1, 1.0), (2, 0.2741976893453145), (3, 0.4684603210490617), (4, 0.41492794149279416), (5, 0.37747055300459176), (6, 0.2942219071251329), (7, 0.0), (8, 0.12468951813214109), (9, 0.21498599439775912), (10, 0.14893617021276595)]


### Results

**Descending order** (Shakespeare → Middlemarch → City of God → Moby Dick → Pride and Prejudice → Frankenstein → A Room with a View → Alice's Adventures in Wonderland → Romeo and Juliet → Romeo and Juliet First Folio):

`[(1, 1.0), (2, 0.5116789778398882), (3, 0.3458348103509394), (4, 0.3794257291431155), (5, 0.1317685047193244), (6, 0.08793324775353016), (7, 0.21521942110177406), (8, 0.1497987349051179), (9, 0.0), (10, 0.40223704022370405)]`

**Randomized order** (Shakespeare → Frankenstein → Moby Dick → Romeo and Juliet First Folio → Middlemarch → City of God → Romeo and Juliet → Pride and Prejudice → A Room with a View → Alice's Adventures in Wonderland):

`[(1, 1.0), (2, 0.2741976893453145), (3, 0.4684603210490617), (4, 0.41492794149279416), (5, 0.37747055300459176), (6, 0.2942219071251329), (7, 0.0), (8, 0.12468951813214109), (9, 0.21498599439775912), (10, 0.14893617021276595)]`

### What this revealed

**Finding 1 — BPE vocabulary is order-independent.** Both runs produced identical results: 332 merges learned, 500 final vocab size, and the same first and last 20 merges regardless of source order. This directly confirms a fundamental BPE property: the learned vocabulary converges to the same result when trained on the same aggregate corpus. This is because the merges are driven by total frequency across all sources, not by the order sources were encountered (with the exception of ties). The order affects the new-word discovery tracking, but not the tokenizer itself — a meaningful distinction between the algorithm's output and the measurement of how information accumulates during training.

**Finding 2 — New-word discovery tapers, but not smoothly.** The general trend confirms the hypothesis: discovery rates are high early and lower later. But the curve is not monotonically decreasing — individual sources spike above their neighbors depending on how different their vocabulary is from what came before. This tells us the discovery rate is driven by **source diversity relative to the accumulated vocabulary**, not just position in the sequence. Findings 3 through 5 examine specific cases that explain why.

**Finding 3 — Complete overlap produces exactly 0.0.** The modern-spelling standalone Romeo and Juliet (`romeo_clean.txt`) produced a new-word discovery rate of exactly 0.0 in both runs, regardless of position. Every word in that edition was already present in Shakespeare's complete works - a clean, precise confirmation both that the tracking mechanism works correctly, and that the complete works genuinely encapsulate the standalone edition with no gaps.

**Finding 4 — "Same work, different edition" can mean something significant: the First Folio case.** The First Folio transcription (`romeo2_clean.txt`) produced approximately 40% new-word discovery in both runs (0.402 descending, 0.415 randomized), despite being a version of the same play already fully contained in the complete works. Examining the file directly revealed two contributing factors:

First, the file contains modern-English editorial notes (Executive Director's Notes and Scanner's Notes) written by the digitization team, introducing contemporary vocabulary not present in the play itself. However, this likely contributed only a modest fraction of the novelty, given how much modern English had already been covered by the preceding sources.

Second, and more substantially, the file is a First Folio transcription explicitly preserving 17th-century printed spelling conventions - including systematic `u`/`v` substitution ("liue" for "live", "vnfold" for "unfold"), period-specific typographical conventions, and spelling variations documented by the original scanner as intentionally kept from the 1623 printed text. To BPE, "liue" and "live" are entirely unrelated tokens. The ~40% novelty is an indicator of **how different 17th-century printed English language is from modern English spelling** as a systematic phenomenon across the entire play's vocabulary.

This has a real practical implication: corpus diversity includes systematic spelling variation across historical periods, not just topical or authorial diversity. A tokenizer trained only on modern-spelling text would handle period-accurate historical texts poorly.

**Finding 5 — Domain and genre diversity matter as much as chronological distance.** Two sources contributed unexpectedly high novelty relative to their position in the sequence: The City of God and A Room with a View.

The City of God (book 3 in the descending run, book 6 in the randomized run) is a 5th-century Latin theological treatise translated into Victorian English. Despite the translation being contemporary with Middlemarch and other Victorian novels in the corpus, its content (Christian philosophy, Roman history, pagan religion, ecclesiastical terminology, and Latin-derived theological vocabulary) introduces an entirely different vocabulary domain from the fiction novels surrounding it. "Same era of translation" does not mean "same vocabulary domain."

A Room with a View (1908) contributed similarly elevated novelty despite being from the most recent era in the corpus. Forster's Edwardian social commentary, Italian setting references, and distinctive prose style introduce vocabulary that simply does not appear frequently in Victorian seafaring novels, Gothic fiction, Romantic-era science fiction, or 18th-century social comedies - regardless of chronological proximity.

Together, these two cases demonstrate that **vocabulary novelty is driven by the intersection of era, domain, genre, and authorial style** and not by any single dimension alone. A production tokenizer aiming for broad generalization would want to deliberately maximize diversity across all of these dimensions, not just ensure chronological spread.

## Section 7 - Observations on Non-English Text

How does this tokenizer behave with languages that don't use spaces between words, use a different character set, or structure text differently? This section trains the tokenizer on an excerpt of *道徳の観念* (Moral Concepts) by 戸坂潤 (Tosaka Jun), a public domain philosophical text from 青空文庫 (Aozora Bunko) - the Japanese equivalent of Project Gutenberg. The text was copied directly from the HTML version of the source page, since the raw file uses ShiftJIS encoding rather than UTF-8.

The code follows the same pattern as Section 5, substituting the Japanese excerpt for the Python code sample.

In [35]:
japanese_text = '''
道徳の観念
戸坂潤



　第一章　道徳に関する通俗常識的観念

　道徳の問題を持ち出す際、いつも邪魔になるものは、道徳に関する世間の通俗常識である。ここで通俗常識というのは、常識があるとか常識がないとかいう、ああした人間の共通な生活必需観念の謂ではなくて、却って世間の人がごく便宜的に大まかに粗雑に振り回している処の、出来合いの観念のことを云うのであるが、この意味に於ける通俗常識は、事物を少し細かく検討しようとする時に、大抵邪魔になる。これは今更ここで説くまでもないことだろう。だが今の場合、事が道徳の問題に関してだと、この邪魔になり方が普通の場合に較べて比較にならぬ程甚だしいのだ、ということを注意したいのである。それはなぜかというと、後に説明するように、道徳そのものが実は或る一定の意味に於ける常識に他ならないからで、常識自身はそこまでつきつめて考えないに拘らず、道徳とは常識そのものと斉しく生活意識［＃「生活意識」に傍点］全般を総括する名称だと考えられねばならぬだろうからである。生活意識全般は、或る一定の意味の常識なのだ。
　尤も道徳というものに関する常識的な観念が、道徳というものに就いての理論的な分析省察の邪魔になるからと云って、この常識自身と全く別な世界にぞくする言葉で道徳を説明するのでは、元来道徳の説明［＃「説明」に傍点］でも何でもなくなって了うだろう。そういう意味では道徳の理論的な観念はいつも道徳の常識的観念を縁とすることによって、その検討が始められねばならず、そして終局に於て、常識的道徳観念からの絶縁としてではなくて却ってそれの深化又は変貌として、道徳に関する理論的概念を取り出さねばならぬ。だがそのためにも、道徳に就いての常識的な観念が、殆んど迷信に近いまでに頑なで有害なものだということを知らねばならぬ。
　常識はまず第一に、道徳というものを社会構造の領域乃至文化領域の一つだと仮定している。と云うのは、社会機構の諸層は常識によると、経済・政治・社会関係・道徳界・芸術・宗教・学問・等々に区分されている。この区分法の原理を吟味して社会構築の段階として之等のものを適当な順序に排列するのだとすれば、この区分をすること自身は科学的なことで誤りではないのだが（史的唯物論の不朽の功績の一つはここにある）、併しそれにも拘らずその場合にも、あくまで道徳に関する通り一遍の常識を利用［＃「利用」に傍点］してそう云っているのであって、道徳なるものに関するこの場合の常識的想定そのものに就いては、なお問題を残しているのである。史的唯物論がそこで［＃「そこで」に傍点］問題にしているのは（併し他の場合には問題がもっと変らねばならぬが）、所謂道徳なるもの（と云うのは「常識的に」道徳と呼ばれている処のもののことだ）が決してそれ自身絶対に独立した全く独自な原則に立つものではなく、実は社会機構に於ける下部構造の上に建てられた処の、そしてこの下部構造を原因とする一つの結果としての、上部構造の一部分に他ならぬ、ということであって、この所謂道徳なるものが実はどういう含蓄を有つものであるかは、その限りではしばらく論外におかれているのである。
　従って、道徳がそうした何か判り切ったような一領域であり、他の諸領域との区別限界などが初めから知れ切ったものであるかどうか、それはまだその限りでは問題ではないのだ。つまり史的唯物論が道徳に対して、そのイデオロギー論的段階づけによって一定の領域を指定した限りでは、さし当り常識で道徳と呼んでいる処のものはここに位置するものだということを、科学的に単に指示したに過ぎないのであって、それ以上に、この常識的な道徳という観念によって指し示された領域が果してそのままで充分に理論的に不都合のないものかどうかは、まだ問題になっていない。――だが史的唯物論によるイデオロギー理論乃至文化理論は、問題を当然そこまで押し進めなければならない筈だ。そうすると、一体道徳とは何かということが初めて根本的に問題になる。一体道徳という観念［＃「観念」に傍点］が何かということからが問題になって来ざるを得ない。道徳なるものの占める領域がどこからどこまでに渡っているかというような領土問題などは、その時、道徳という観念の如何に対応する名目的な問題になると云うことが出来るかも知れない。
　道徳の領域は常識によると大して問題にならない程度に判然としているように思われている。例えば法律で禁じられていないに拘らず道徳では禁じられている行為がある。これで見ると恐らく道徳の領域は法律の領域よりも広く、そして又恐らく之を含んだものだろう、という風に考えられる。法律で禁じられていても道徳的には正しいと意識される場合も、今日のブルジョア的乃至半封建的法律では決して少なくないが、それは元来道徳そのものが二つに（階級的に）分裂しているからで、一方の側の道徳から見て善い行為も、他の側の道徳から見て悪いということになっていればこそ、法律上でも禁止されているわけだし、それにこの点をもっと便宜的に片づけるには、悪法も法である以上之に従うことが道徳的だという風に形式化して考えれば、咄は極めて簡単だ。とに角道徳界と法律界との限界は判然としているように見るのが、常識である。
　経済領域と道徳領域との区分も亦、常識的には一応判然としている。物質への興味と精神への興味とは相容れない裏表であると考えられている。カエサルのものはカエサルへ、神のものは神へ、と云うのである。史的唯物論は生産関係によって経済関係乃至社会関係を説明するのであるが、道徳も亦この生産関係から終局的に説明される。それはすでに云ったことだが、その場合にも依然として経済関係（乃至社会関係）と道徳との常識的限界を利用している。と同じに道徳と政治との限界さえが常識を利用して設定されている。尤もこの常識を利用したからと云って、この理論自身が常識を仮定しているということにもならず、ましてこの理論が常識的だということにもなるのではないが、にも拘らずここで常識的限界が利用されているという事実は今大切だ。
　史的唯物論を模倣した一例はＦ・シュタウディンガーの著書『道徳の経済的基礎』（岩波文庫版）である。之によると経済が道徳を決定するというのであるが、処がこの道徳なるものは、要するに単に社会秩序乃至社会機構のことに他ならないのである。人間相互の物的関係や利益社会関係や共同社会関係が道徳の材料であって、この共同社会関係が他の社会関係の上位に位するようになることが、取りも直さず道徳ということだと考えられている。従って道徳は、もはや経済機構自身や社会機構自身と領域的に別なものではないので、社会全体が道徳的本質に他ならぬものとなる。社会主義も亦一つの倫理学に帰着する。政治も亦道徳に他ならない、ということになっているわけである。――社会を道徳に還元することは独りシュタウディンガーに限らず、多くのカント社会主義者（乃至カント主義的マルクス主義者）の共通特色であって、一見之は、道徳や倫理を、もはや常識的な狭い領域にとじこめられた観念としてではなく、之を最も広範な含蓄を持った観念にまで深化するもののように見えるかも知れない。だが実は、これこそ何よりも、道徳の独立領域［＃「独立領域」に傍点］という常識観念の誇張の結果そのものなのだ。
　カントは経験界とは全く独立な之とは全く絶縁した本体界を、英知界を、道徳の世界・道徳の領域と考えた。之は道徳という領域が何かハッキリと決って他の領域から機械的に限界されて横たわっているという一つの根本的な常識を、批判体系の根柢として採用したことであって、シュタウディンガーやＭ・アードラー、フォルレンダー達のカント社会主義者は、多かれ少なかれ、この常識のこうした科学的合法化の後継者に他ならなかったのである（この点に関しては米田庄太郎『輓近社会思想の研究』上巻が参考となる）。
　道徳の領域が何かハッキリしているように想定されるのは、実は道徳に関する観念自身が機械的に固定しているからなので、道徳という観念と他領域の観念との間に機械的に限界を引き得るとか、道徳という観念は固定不動なものだとか、考えることに由来する。そしてこういう考え方は要するに道徳の内容そのものが固定不動なものだという考え方から脈を引いているのである。――なる程道徳と名づけられる一つの領域が存することを、吾々は何としても疑うことは出来ないだろう。だが夫は何も道徳という独立な世界がどこかでハッキリとした柵をめぐらしているということにはならぬ。問題はいつも道徳の領域と他領域との交流［＃「交流」に傍点］であり而もその本質的な交流なのだ。道徳と政治との交流はシュタウディンガーも触れているが、卑俗な形では現に政治の倫理化とか政教一致とかなって現われている。特に道徳と法律との交流は著しいので、ヘーゲルなどは両者を「抽象法」の名の下に一緒に取り扱っていると云ってもよい。アメリカの法律家ロスコー・パウンドは実際家の見地から、この交流に就いて興味深い分析を加えている（『法と道徳』高柳・岩田訳）。
　だがそれより以上に大切なことは、一体道徳なるものが、一般に一つの領域だ（その限界は機械的に与えるべからざるものでその内容も固定不変なものではないとして）と云って片づけられ得るかどうかなのだ。と云うのは、道徳は社会関係・政治関係・法律体系・其の他其の他と並列［＃「並列」に傍点］する一領域であると考えられるにも拘らず、他方之等一切の諸領域の一つ一つに接着していることをも見落すことが出来ないのである。そういう関係があればこそ、社会そのものが道徳的本質に還元されたり、政治や法律が単なる道徳に帰着されたりするということも初めて可能だったわけで、社会主義が倫理学に包括されて了うという誤りも、決して理由なしには発生しなかったのである。でもしそうだとすると、道徳はもはや単なる一領域であるに止まらず、恐らく一領域であるにも拘らず他領域をも蔽うか又は之に付随するかする処の、或るものだと云わねばならぬ。之をなお或る種の領域だと云うことは自由だが、それはもはや之まで云って来た意味での一領域ではない。
　だから、例はやや飛躍するが、プラトンが善のイデアを最高のイデア、諸イデアのピラミットの頂点と考えたことには意味があったわけで、善のイデアはもはや他の諸イデアと並列したものではなく、一段と高いオーダーにぞくすることを意味するのだと解釈すべきだとも云われている。だがそうだからと云って誰もプラトンを汎道徳主義者や倫理主義者に数えようとはしないだろう。――つまり凡てのものを道徳に還元しようという各種の汎道徳主義乃至倫理主義なるものは、実は凡ての他領域の事物を、道徳という一つの領域［＃「領域」に傍点］に還元しようとするからこそ誤っているのであって、之は、道徳というものをどこまでも一領域にすぎぬものと仮定しておいた上で、さてこの特別な一領域を無遠慮にも世界の全領域に押し拡げよう、という仕組みに他ならない。常識はいつでもこの種の手口を便宜に思うもので、教育学者や教育家は、教育というものを何か自分達の専門［＃「専門」に傍点］の領域だと仮定した上で、この専門領域の内に世の中の一切のものを抛り込んで了おうとする。徳育や修身の専門家（？）が養成されるのも、日本のこの教育専門家の領域［＃「領域」に傍点］からであるが、そうした道徳の専門家の考えは、道徳が生活の特別な一領域であり従って［＃「従って」に傍点］又生活の全領域を含む領域だという、道徳領域説の常識が、漫画になったものだ。
　道徳が生活場面の一領域の意味に尽きると考えることは、道徳に就いての常識的観念の第一の不備な点であった。之によるとスッカリ道徳にぞくするものと全く道徳にぞくさないものとが、ハッキリ区別されるわけだが、併し之ほど都合の悪い結果を伴うものはあるまい。芸術は芸術であって道徳とは別だという。それでは芸術に於ける道徳＝モラル程、無意味なものはなくなるわけだ。内容は道徳でも形式は道徳ではなくて芸術だと云うのだろうか。だが道徳の形式とは何だろう。それはもはや常識が答え得る問いではないかも知れないが、自由意志の自律に従うとか、目的意識的行為の形を取ったとかいうことだろう。処が自律に従わなかった場合も決して道徳の領域外にあったのではなくて、却って反道徳・不道徳という刻印を捺されるために、あくまで道徳の領域の内になければならぬ。実践的行為の形を取るということが道徳の形式でないことは、芸術的創作だって実践的行為だし、単に善いか悪いか何かを考えただけでも立派に道徳的な問題にぞくする、そういうことを考えて見れば判ることだ。――道徳なるものは、だから生活の一切の領域に、或る仕方に於て着き得るのだ。どういう権利でどういう仕方で着くかは、後に見ようと思うが、とに角その意味に於て、道徳とは生活意識そのものを意味するのだと、仮に云っておくこととしよう。念のため断わっておくが、道徳は確かに一応、常識がそう想定している通り、生活の一領域のことなのだ。にも拘らず、それに尽きることなく［＃「それに尽きることなく」に傍点］、根本的には生活意識そのものを意味するという含蓄を有つものだ、と云うのである。
'''

In [36]:
words = japanese_text.split()
print(words[:20])

['道徳の観念', '戸坂潤', '第一章', '道徳に関する通俗常識的観念', '道徳の問題を持ち出す際、いつも邪魔になるものは、道徳に関する世間の通俗常識である。ここで通俗常識というのは、常識があるとか常識がないとかいう、ああした人間の共通な生活必需観念の謂ではなくて、却って世間の人がごく便宜的に大まかに粗雑に振り回している処の、出来合いの観念のことを云うのであるが、この意味に於ける通俗常識は、事物を少し細かく検討しようとする時に、大抵邪魔になる。これは今更ここで説くまでもないことだろう。だが今の場合、事が道徳の問題に関してだと、この邪魔になり方が普通の場合に較べて比較にならぬ程甚だしいのだ、ということを注意したいのである。それはなぜかというと、後に説明するように、道徳そのものが実は或る一定の意味に於ける常識に他ならないからで、常識自身はそこまでつきつめて考えないに拘らず、道徳とは常識そのものと斉しく生活意識［＃「生活意識」に傍点］全般を総括する名称だと考えられねばならぬだろうからである。生活意識全般は、或る一定の意味の常識なのだ。', '尤も道徳というものに関する常識的な観念が、道徳というものに就いての理論的な分析省察の邪魔になるからと云って、この常識自身と全く別な世界にぞくする言葉で道徳を説明するのでは、元来道徳の説明［＃「説明」に傍点］でも何でもなくなって了うだろう。そういう意味では道徳の理論的な観念はいつも道徳の常識的観念を縁とすることによって、その検討が始められねばならず、そして終局に於て、常識的道徳観念からの絶縁としてではなくて却ってそれの深化又は変貌として、道徳に関する理論的概念を取り出さねばならぬ。だがそのためにも、道徳に就いての常識的な観念が、殆んど迷信に近いまでに頑なで有害なものだということを知らねばならぬ。', '常識はまず第一に、道徳というものを社会構造の領域乃至文化領域の一つだと仮定している。と云うのは、社会機構の諸層は常識によると、経済・政治・社会関係・道徳界・芸術・宗教・学問・等々に区分されている。この区分法の原理を吟味して社会構築の段階として之等のものを適当な順序に排列するのだとすれば、この区分をすること自身は科学的なことで誤りではないのだが（史的唯物論の不朽の功績の一つはここにある）、併しそれにも拘らずその場合にも、あくまで道徳

In [37]:
code_merges, code_vocab = train_bpe(japanese_text, target_vocab_size=600)
print(f"Number of merges learned: {len(code_merges)}")
print(f"Final vocab size: {len(code_vocab)}")

print("\nFirst 15 merges:")
for m in code_merges[:15]:
    print(m)

print("\nLast 15 merges:")
for m in code_merges[-15:]:
    print(m)

Number of merges learned: 79
Final vocab size: 600

First 15 merges:
('道', '徳')
('も', 'の')
('い', 'う')
('領', '域')
('こ', 'と')
('と', 'いう')
('す', 'る')
('し', 'て')
('常', '識')
('っ', 'て')
('で', 'あ')
('い', 'る')
('な', 'い')
('は', '、')
('であ', 'る')

Last 15 merges:
('主', '義')
('なる', 'もの')
('だ', 'ろ')
('だろ', 'う')
('だ', 'が')
('の', 'である。')
('その', 'もの')
('ら', 'ず')
('に', 'よ')
('と', 'して')
('さ', 'れ')
('る', 'と')
('場', '合')
('問題', 'に')
('自', '身')


### Observation 1 — target_vocab_size must account for base character set size

I had initially run `train_bpe` with `target_vocab_size=150`, which produced 0 merges and a final vocab size of 521. The 0 merges were due to the `while len(vocab) < target_vocab_size` condition evaluating to `False` immediately. This was because the 521 unique Japanese characters found in the text already exceeded the target vocabulary size before the merge loop ever ran.

This is a meaningful difference from English: the Latin alphabet requires roughly 60-80 base characters (letters, digits, punctuation), leaving most of the `target_vocab_size` budget available for learned merges. Japanese, on the other hand, uses three scripts (kanji, hiragana, and katakana) and requires hundreds of base characters to represent the text. Setting `target_vocab_size=600` (safely above 521) produced 79 merges and confirmed the algorithm actually works.

This has a direct practical implication: for multilingual tokenizers, `target_vocab_size` must be set with the base character requirements of all target languages in mind and not just the dominant language. This is one reason why byte-level BPE uses exactly 256 base tokens regardless of language, leaving the full remaining budget available for learned merges across any script.

### Observation 2 — Without language-specific pre-tokenization, entire paragraphs become single "words"

Japanese does not use spaces between words. When `text.split()` runs on Japanese prose separated only by newlines, it produces one entry ('word') per paragraph with each appearing exactly once. This eliminates the key efficiency advantage of word-level pre-splitting: instead of a compact dictionary of thousands of repeated words with accumulated frequency counts, `word_freqs` contains a handful of unique, unrepeated paragraph-length strings. The mechanism itself continues to accumulate frequency across all entries, but the scale here is limited to ~12–13 unique paragraph-strings - an artifact of this being an excerpt rather than the full book. A full book would produce more entries but the structural limitation remains unchanged.

This is a different failure mode from the Section 5 Python code experiment. There, the problem was a small and repetitive sample in a language the tokenizer could pre-tokenize correctly - too much repetition of the same narrow patterns led to runaway merging. Here, the issue is the opposite: the tokenizer lacks the pre-tokenization strategy Japanese requires, so instead of many short, repeated word-entries, it gets long, unique paragraph-strings. Languages like Japanese, Chinese, and Thai need language-specific word segmenters (such as MeCab for Japanese) to identify word boundaries before BPE training begins - the same role whitespace-splitting plays for European languages.

Further research revealed that Japanese word segmenters like MeCab make context-dependent decisions about whether character sequences represent one compound word or two separate morphemes - for example, whether 道徳 is one word meaning "morality" or two separate characters meaning "road" and "virtue." This adds a layer of ambiguity that English whitespace-splitting avoids entirely, and illustrates why some production multilingual tokenizers bypass language-specific pre-tokenization entirely in favor of byte-level BPE.

### Observation 3 — Despite this limitation, BPE still discovers meaningful units purely from character frequency

Despite the pre-tokenization mismatch, rerunning with `target_vocab_size=600` produced genuinely interesting merges. The first 15 merges learned:

- `('道', '徳')` → 道徳 ("morality/ethics") - the core topic of this philosophical text, appearing constantly throughout every paragraph
- `('も', 'の')` → もの ("thing/matter") - extremely common in Japanese prose
- `('い', 'う')` → いう ("to say/called") - also extremely common
- `('領', '域')` → 領域 ("domain/territory") - a key term in this specific text about the domains of morality
- `('こ', 'と')` → こと ("thing/fact") - one of the most common words in Japanese

The algorithm had no knowledge of Japanese linguistics, yet independently discovered that 道徳, 領域, and もの are high-value units to tokenize together purely based ont character frequency. This is the same phenomenon as "the" assembling itself from `('t','h')` and `('e','▁')` in the Alice's Adeventures in Wonderland experiment: data-driven, no linguistic rules, yet producing linguistically meaningful results.

The last 15 merges are equally interesting. `('の', 'である。')` combines a grammatical particle with a formal copula ending and a period. This pair was discovered as a reusable unit because this philosophical text consistently uses formal written Japanese throughout. `('なる', 'もの')` → なるもの ("what is called/so-called") appeared as a recurring philosophical framing device. These longer merged units begin to approach phrase-level patterns - a reminder that as merge depth increases, the tokenizer starts capturing constructs whose *meaning* is better understood by the transformer than by the tokenizer itself.

### What this illustrated

This experiment surfaced three practical lessons that don't appear in the English experiments:

First, `target_vocab_size` implicitly determines whether BPE training can run at all, depending on the base character requirements of the target language.

Second, the effectiveness of BPE depends critically on the pre-tokenization strategy matching the linguistic structure of the input. A tool designed for English whitespace-splitting is not a universal solution. It requires language-specific adaptation to be effective on languages without spaces.

Third, BPE's core frequency-counting mechanism is genuinely language-agnostic: given any Unicode text, it will find and merge the most frequent adjacent symbol pairs, regardless of whether those symbols are Latin letters, kanji, or hiragana.